# 09 — Widgets & Interactivity

ipywidgets bikin notebook lo bisa diinteraksiin — slider, dropdown, tombol, tanpa perlu web framework.

**Yang akan lo pelajari:**
- Install dan setup ipywidgets
- Widget dasar: slider, text, dropdown, checkbox
- `interact` dan `interactive` decorator
- Output widget
- Layout dan styling
- Contoh aplikasi interaktif

---

In [ ]:
# Install kalau belum ada
# !pip install ipywidgets

import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
print(f"ipywidgets version: {widgets.__version__}")

## 1. Widget Dasar

In [ ]:
# IntSlider
slider = widgets.IntSlider(
    value=50,
    min=0, max=100, step=1,
    description='Nilai:',
    continuous_update=True
)
display(slider)

# Akses value programatik
print(f"Nilai awal: {slider.value}")

In [ ]:
# FloatSlider
float_slider = widgets.FloatSlider(
    value=0.5, min=0.0, max=1.0, step=0.01,
    description='Rate:',
    readout_format='.2f'
)

# Text input
text_input = widgets.Text(
    value='Hello',
    placeholder='Ketik sesuatu...',
    description='Input:'
)

# Dropdown
dropdown = widgets.Dropdown(
    options=['Linear', 'Polynomial', 'Exponential'],
    value='Linear',
    description='Model:'
)

# Checkbox
checkbox = widgets.Checkbox(
    value=True,
    description='Tampilkan grid'
)

# Tombol
button = widgets.Button(
    description='Klik Saya',
    button_style='primary',  # '', 'success', 'info', 'warning', 'danger'
    icon='check'
)

display(float_slider, text_input, dropdown, checkbox, button)

In [ ]:
# Widget lainnya
color_picker = widgets.ColorPicker(
    value='#2196F3',
    description='Warna:'
)

date_picker = widgets.DatePicker(
    description='Tanggal:'
)

progress = widgets.IntProgress(
    value=65, min=0, max=100,
    description='Progress:',
    bar_style='info'
)

select_multiple = widgets.SelectMultiple(
    options=['NumPy', 'Pandas', 'Matplotlib', 'Scikit-learn', 'TensorFlow'],
    description='Library:',
    rows=5
)

display(color_picker, date_picker, progress, select_multiple)

## 2. interact — Cara Paling Simple

In [ ]:
from ipywidgets import interact, interactive, fixed

# interact otomatis buat widget berdasarkan tipe argumen
# int/float → slider
# bool → checkbox
# str → text input
# list/tuple → dropdown

def sapa(nama="Dunia", formal=False):
    if formal:
        print(f"Selamat pagi, {nama}.")
    else:
        print(f"Halo, {nama}!")

interact(sapa, nama="Budi", formal=False);

In [ ]:
# interact dengan plot — tiap slider berubah, plot update otomatis
@interact(
    frekuensi=(0.5, 5.0, 0.1),
    amplitudo=(0.1, 2.0, 0.1),
    fase=(0.0, 2*np.pi, 0.1),
    tipe=['sin', 'cos', 'tan']
)
def plot_gelombang(frekuensi=1.0, amplitudo=1.0, fase=0.0, tipe='sin'):
    x = np.linspace(0, 4*np.pi, 500)
    
    if tipe == 'sin':
        y = amplitudo * np.sin(frekuensi * x + fase)
    elif tipe == 'cos':
        y = amplitudo * np.cos(frekuensi * x + fase)
    else:
        y = np.clip(amplitudo * np.tan(frekuensi * x + fase), -5, 5)
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(x, y, color='#2196F3', linewidth=2)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.set_ylim(-3, 3)
    ax.set_title(f'{tipe}(x) — A={amplitudo}, f={frekuensi}, φ={fase:.2f}')
    ax.grid(True, alpha=0.3)
    plt.show()

## 3. Output Widget — Kontrol di Mana Output Muncul

In [ ]:
output = widgets.Output()

button = widgets.Button(
    description='Generate Plot',
    button_style='primary'
)

counter = {'n': 0}

def on_button_click(b):
    counter['n'] += 1
    with output:
        clear_output(wait=True)  # hapus output sebelumnya
        fig, ax = plt.subplots(figsize=(8, 4))
        data = np.random.randn(200)
        ax.hist(data, bins=30, color=np.random.choice(['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']))
        ax.set_title(f'Histogram Random #{counter["n"]}')
        plt.show()

button.on_click(on_button_click)

display(button, output)

## 4. Layout Widget

In [ ]:
# HBox — horizontal layout
w1 = widgets.IntSlider(description='X:', min=0, max=100)
w2 = widgets.IntSlider(description='Y:', min=0, max=100)
w3 = widgets.IntSlider(description='Z:', min=0, max=100)

hbox = widgets.HBox([w1, w2, w3])
display(hbox)

print("---")

# VBox — vertical layout
label = widgets.Label(value='Pilih opsi:')
dd = widgets.Dropdown(options=['A', 'B', 'C'])
btn = widgets.Button(description='Submit', button_style='success')

vbox = widgets.VBox([label, dd, btn])
display(vbox)

## 5. Contoh Aplikasi — Kalkulator BMI Interaktif

In [ ]:
# Kalkulator BMI lengkap dengan UI
output_bmi = widgets.Output()

# Widgets
slider_tinggi = widgets.IntSlider(value=170, min=140, max=220, description='Tinggi (cm):')
slider_berat = widgets.FloatSlider(value=65, min=30, max=150, step=0.5, description='Berat (kg):')
btn_hitung = widgets.Button(description='Hitung BMI', button_style='primary', icon='calculator')

def hitung_bmi(b):
    tinggi_m = slider_tinggi.value / 100
    berat = slider_berat.value
    bmi = berat / (tinggi_m ** 2)
    
    if bmi < 18.5:
        kategori, color = 'Underweight', '#FF9800'
    elif bmi < 25:
        kategori, color = 'Normal', '#4CAF50'
    elif bmi < 30:
        kategori, color = 'Overweight', '#FF9800'
    else:
        kategori, color = 'Obese', '#F44336'
    
    with output_bmi:
        clear_output(wait=True)
        
        fig, ax = plt.subplots(figsize=(8, 3))
        ranges = [18.5, 25, 30, 40]
        ax.barh(['BMI'], [18.5], color='#FF9800', alpha=0.3, label='Underweight')
        ax.barh(['BMI'], [6.5], left=18.5, color='#4CAF50', alpha=0.3, label='Normal')
        ax.barh(['BMI'], [5], left=25, color='#FF9800', alpha=0.3, label='Overweight')
        ax.barh(['BMI'], [10], left=30, color='#F44336', alpha=0.3, label='Obese')
        ax.axvline(bmi, color=color, linewidth=3, label=f'BMI lo: {bmi:.1f}')
        ax.set_xlim(10, 40)
        ax.legend(loc='upper right')
        ax.set_title(f'BMI: {bmi:.1f} — {kategori}', fontsize=14, color=color, fontweight='bold')
        ax.set_xlabel('BMI')
        plt.tight_layout()
        plt.show()

btn_hitung.on_click(hitung_bmi)

ui = widgets.VBox([
    widgets.HTML('<h3>🏃 Kalkulator BMI</h3>'),
    slider_tinggi,
    slider_berat,
    btn_hitung,
    output_bmi
])

display(ui)

## 6. Contoh Aplikasi — Eksplorasi Dataset

In [ ]:
import pandas as pd

# Buat dataset simulasi
np.random.seed(42)
n = 300
df = pd.DataFrame({
    'umur': np.random.randint(18, 60, n),
    'gaji': np.random.lognormal(mean=10, sigma=0.5, size=n),
    'pengalaman': np.random.randint(0, 20, n),
    'divisi': np.random.choice(['Engineering', 'Marketing', 'Finance', 'HR'], n)
})

output_explorer = widgets.Output()

dd_x = widgets.Dropdown(options=df.columns.tolist(), value='umur', description='X-axis:')
dd_y = widgets.Dropdown(options=df.columns.tolist(), value='gaji', description='Y-axis:')
dd_divisi = widgets.SelectMultiple(
    options=df['divisi'].unique().tolist(),
    value=df['divisi'].unique().tolist(),
    description='Divisi:',
    rows=4
)

def update_plot(change=None):
    x_col = dd_x.value
    y_col = dd_y.value
    selected = list(dd_divisi.value)
    
    df_filtered = df[df['divisi'].isin(selected)]
    
    with output_explorer:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(9, 5))
        colors = {'Engineering': '#2196F3', 'Marketing': '#4CAF50',
                  'Finance': '#FF9800', 'HR': '#9C27B0'}
        
        for divisi in selected:
            mask = df_filtered['divisi'] == divisi
            ax.scatter(df_filtered[mask][x_col],
                       df_filtered[mask][y_col],
                       label=divisi, alpha=0.6, s=50,
                       color=colors.get(divisi, 'gray'))
        
        ax.set_xlabel(x_col)
        ax.set_ylabel(y_col)
        ax.set_title(f'{x_col} vs {y_col} (n={len(df_filtered)})')
        ax.legend()
        plt.tight_layout()
        plt.show()

# Observe changes
dd_x.observe(update_plot, names='value')
dd_y.observe(update_plot, names='value')
dd_divisi.observe(update_plot, names='value')

# Initial render
update_plot()

controls = widgets.HBox([widgets.VBox([dd_x, dd_y]), dd_divisi])
display(widgets.VBox([
    widgets.HTML('<h3>📊 Dataset Explorer</h3>'),
    controls,
    output_explorer
]))

---
## ➡️ Next
Lanjut ke **[10_debugging_profiling.ipynb](10_debugging_profiling.ipynb)** — cara debug error dan ukur performa kode.